# Quadcopter Cascade PID — Figure-8 Trajectory Tracking

This notebook implements a **Cascade PID Controller** for quadcopter trajectory tracking using [c4dynamics](https://c4dynamics.github.io/c4dynamics/).

A nonlinear 6-DOF rigid body model tracks a figure-8 reference trajectory using three nested PID control loops running at different rates.

---

## What you can modify

| Section | What to change |
|---------|----------------|
| **Vehicle** | Mass, inertia, arm length |
| **Trajectory** | Figure-8 size, speed, altitude, duration |
| **Controller** | PID gains for each loop |

After modifying parameters, run all cells top to bottom.

---

## How to use this notebook

1. Read each background section to understand the model and controller.
2. Edit parameters in the **User Inputs** cell that follows each section.
3. Run **Section 4 — Run Simulation** (one cell).
4. View results in **Section 5 — Results**.

All implementation details (dynamics, PID class, simulation loop) are in `quad_pid_utils.py`.

In [ ]:
# Install c4dynamics if running on Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install c4dynamics -q

from quad_pid_utils import run_fig8_pid, plot_results, compute_metrics

---
## Section 1 — Quadcopter Rigid-Body Dynamics

### Model Overview

The quadcopter is modeled as a **nonlinear 6-DOF rigid body** described by 12 states:

$$\mathbf{X} = [x,\ y,\ z,\ \dot{x},\ \dot{y},\ \dot{z},\ \phi,\ \theta,\ \psi,\ p,\ q,\ r]^T$$

The model combines two coupled sets of equations:

- **Translational dynamics** — Newton's second law applied in the inertial frame. The total thrust force $F$ acts along the body z-axis and is projected onto the inertial frame via the rotation matrix $R(\phi, \theta, \psi)$.
- **Rotational dynamics** — Euler's equations of rigid-body rotation. The three torques $(\tau_\phi,\ \tau_\theta,\ \tau_\psi)$ drive the angular accelerations, with inertia cross-coupling terms between roll, pitch, and yaw.

The four control inputs are:

$$\mathbf{U} = [F,\ \tau_\phi,\ \tau_\theta,\ \tau_\psi]^T$$

No linearization or small-angle approximation is used — the full Newton-Euler equations are integrated numerically.

For the full theoretical background on rigid-body kinematics and the state representation used by c4dynamics, see the [c4dynamics kinematics concept page](https://c4dynamics.github.io/c4dynamics/concepts/kinematics.html).

### Reference Frames


The rotation from body to inertial frame uses the ZYX Euler angle convention
$(\psi \rightarrow \theta \rightarrow \phi)$, which is standard for aerial vehicles.

Thrust force $F$ acts upward along $-Z_b$. The four motor thrusts combine to produce
the net force and three torques via the allocation matrix.

### Implementation

The rigid-body dynamics and numerical integration are implemented in `quad_pid_utils.py`:

| Function | Role |
|----------|------|
| `quad_dynamics()` | Computes the 12 state derivatives from current state and control inputs |
| `solve_ivp(..., method='RK45')` | SciPy integrator called inside `run_fig8_pid()` to advance the state one timestep — no separate wrapper function |

The c4dynamics `rigidbody` object stores and updates the vehicle state throughout the simulation.

---
### Vehicle Parameters

Modify the dictionary below to match your vehicle's physical properties.

In [ ]:
# ── Vehicle Parameters ──
# Physical properties of the quadcopter.
# Change these to match your specific vehicle.

vehicle = {
    'm'  : 1.0,     # mass [kg]
    'g'  : 9.81,    # gravity [m/s²]
    'Ixx': 0.0196,  # roll  moment of inertia [kg.m²]
    'Iyy': 0.0196,  # pitch moment of inertia [kg.m²]
    'Izz': 0.0264,  # yaw   moment of inertia [kg.m²]
    'l'  : 0.225,   # arm length — center to motor [m]
    'kT' : 1.0,     # thrust coefficient (normalized)
    'kQ' : 0.01,    # torque coefficient (normalized)
}

---
## Section 2 — Cascade PID Control

### Why Cascade PID for a Quadcopter?

A quadcopter is an **underactuated system** — it has 6 degrees of freedom but only 4 independent control inputs.
Position and attitude are tightly coupled: to move horizontally, the vehicle must first tilt.
This coupling makes a single PID loop inadequate; a cascade (nested-loop) structure is the natural solution.

The key insight is that **attitude dynamics are much faster than position dynamics**.
By separating them into inner and outer loops running at different rates, each loop only has to deal
with the dynamics it can actually influence at its own bandwidth.

### 3-Loop Architecture


| Loop | Rate | Input | Output |
|------|------|-------|--------|
| **Outer** — Position | 50 Hz | Position error $(x, y, z)$ | Desired angles $\phi_d, \theta_d$ + total thrust $F$ |
| **Middle** — Attitude | 100 Hz | Angle error $(\phi, \theta, \psi)$ | Desired body rates $(p_d, q_d, r_d)$ |
| **Inner** — Rate | 200 Hz | Rate error $(p, q, r)$ | Torques $(\tau_\phi, \tau_\theta, \tau_\psi)$ |

**Tuning order:** always tune inner first, then middle, then outer.
Each loop must be stable before the next outer loop is closed around it.

### Control Law

Each PID controller implements:

$$u = K_p\, e + K_i \int e\, dt - K_d\, \dot{m}$$

where $\dot{m}$ is the **derivative of the measurement** (not the error).
This eliminates derivative kick when the reference changes suddenly.

**Key features of each controller:**
- **Derivative on measurement** — no spike at setpoint step changes
- **First-order derivative filter** — suppresses high-frequency noise amplification
- **Integrator anti-windup** — clamps the integral when the output saturates
- **Velocity feedforward** (outer loop only) — reduces phase lag at trajectory turns

### Implementation

| Class / Function | Role |
|------------------|------|
| `PID` | Generic PID class with anti-windup, derivative filter, and derivative-on-measurement |
| `build_controllers()` | Instantiates all nine `PID` objects and configures their gains, rates, and limits |
| `run_fig8_pid()` | Top-level function that runs the full simulation — the cascade loop logic lives directly inside this function |

---
### Controller Parameters

Default gains below are tuned for the default vehicle. If you change vehicle mass or inertia significantly, retune starting from the inner loop.

In [ ]:
# ── Controller Parameters (Advanced) ───
# Default gains are tuned for the default vehicle above.
# If you change the vehicle, you may need to retune these.
#
# Tuning order: inner first, then middle, then outer.

controller = {

    # ── Inner loop — angular rate (200 Hz) ──
    # Controls P, Q, R body rates → outputs torques
    'Kp_p': 0.80,  'Ki_p': 0.0001,  'Kd_p': 0.010,   # roll  rate
    'Kp_q': 0.80,  'Ki_q': 0.0001,  'Kd_q': 0.010,   # pitch rate
    'Kp_r': 0.60,  'Ki_r': 0.0001,  'Kd_r': 0.008,   # yaw   rate

    # ── Middle loop — attitude (100 Hz) ──
    # Controls Phi, Theta, Psi Euler angles → outputs desired rates
    'Kp_phi'  : 7.0,  'Ki_phi'  : 0.0001,  'Kd_phi'  : 0.90,  # roll
    'Kp_theta': 7.0,  'Ki_theta': 0.0001,  'Kd_theta': 0.90,  # pitch
    'Kp_psi'  : 4.0,  'Ki_psi'  : 0.5,     'Kd_psi'  : 0.40,  # yaw

    # ── Outer loop — position (50 Hz) ──
    # Controls X, Y, Z position → outputs desired angles + thrust
    'Kp_x': 1.00,  'Ki_x': 0.01,  'Kd_x': 0.90,  # X position
    'Kp_y': 1.10,  'Ki_y': 0.01,  'Kd_y': 0.80,  # Y position
    'Kp_z': 10.0,  'Ki_z': 0.50,  'Kd_z': 1.50,  # altitude

    # ── Velocity feedforward gains ──
    # Added to position PID output to reduce phase lag.
    # Increase if actual path lags behind reference.
    # Decrease if actual path overshoots reference.
    'Kff_x': 0.2479,
    'Kff_y': 0.35,
}

---
## Section 3 — Figure-8 Trajectory

### Trajectory Definition

The reference trajectory is a **Lissajous figure-8** in the horizontal plane at a fixed altitude:

$$x_{ref}(t) = A \sin(\omega\, t)$$
$$y_{ref}(t) = B \sin(2\omega\, t)$$
$$z_{ref}(t) = z_{ref} \quad (\text{constant})$$

This is a standard benchmark for trajectory tracking controllers because it combines
simultaneous motion in both horizontal axes, continuous direction reversals, and a
non-trivial curvature profile — all of which stress-test the controller's ability to
reject coupling errors and follow a dynamic reference.

### Parameter Meanings

| Parameter | Physical meaning |
|-----------|------------------|
| `A` | Half-width of the figure-8 in the X direction [m]. Larger values mean a wider trajectory. |
| `B` | Half-width in the Y direction [m]. Controls the "height" of each lobe of the figure-8. |
| `omega` | Angular frequency [rad/s]. Controls how fast the quadcopter flies the path. Period = $2\pi/\omega$. |
| `z_ref` | Constant hover altitude [m]. The vehicle holds this height throughout the maneuver. |
| `t_end` | Total simulation duration [s]. Should be long enough to complete at least one full figure-8 cycle. |

### Soft Start

A cosine ramp is applied to the reference velocity for the first 12 seconds of flight.
Without it, the trajectory starts with an instantaneous velocity discontinuity that would
cause a large transient overshoot in the outer position loop.
The ramp smoothly brings the vehicle from hover to the full trajectory speed.

---
### Trajectory Parameters

Modify the values below to change the shape, speed, and duration of the figure-8.

In [ ]:
# ── Trajectory Parameters ───
# Define the figure-8 shape, speed, and flight conditions.
#
# The figure-8 is defined by:
#   x(t) = A * sin(omega * t)
#   y(t) = B * sin(2 * omega * t)
#
# A cosine ramp is applied for the first 12 seconds of flight
# to avoid a velocity discontinuity at trajectory start.

trajectory = {
    'A'    : 1.5,   # figure-8 X amplitude [m]
    'B'    : 1.0,   # figure-8 Y amplitude [m]
    'omega': 0.3,   # angular frequency [rad/s]
                    # period = 2*pi/omega ≈ 20.9 s per cycle
    'z_ref': 1.5,   # constant hover altitude [m]
    't_end': 40.0,  # total simulation duration [s]
}

---
## Section 4 — Simulation

### What Happens During the Simulation

The simulation advances the vehicle state forward in discrete timesteps of `dt = 0.005 s`
(the inner loop rate, 200 Hz). At each timestep the following steps occur:

1. **Reference generation** — compute the desired position $(x_d, y_d, z_d)$ and
   reference velocity from the figure-8 equations at the current time.
2. **Outer loop (50 Hz)** — every 4th master step, the position PID runs.
   It compares the current position to the reference and outputs the desired
   roll angle $\phi_d$, pitch angle $\theta_d$, and total thrust $F$.
   Velocity feedforward is added to the X and Y outputs.
3. **Middle loop (100 Hz)** — every 2nd master step, the attitude PID runs.
   It compares the current Euler angles to the desired angles from the outer loop
   and outputs desired body rates $(p_d, q_d, r_d)$.
4. **Inner loop (200 Hz)** — every master step, the rate PID runs.
   It compares the current body rates to the desired rates from the middle loop
   and outputs the three torques $(\tau_\phi, \tau_\theta, \tau_\psi)$.
5. **Dynamics integration** — the four control inputs
   $[F,\ \tau_\phi,\ \tau_\theta,\ \tau_\psi]$ are passed to `quad_dynamics()`
   which computes the 12 state derivatives. SciPy's `solve_ivp(..., method='RK45')`
   then integrates the state forward by one timestep — there is no separate wrapper function.
6. **Logging** — the current state, reference, control inputs, and Euler angles
   are appended to the results arrays.

### Implementation

The entire loop above is encapsulated in `run_fig8_pid()` from `quad_pid_utils.py`.
It accepts the three parameter dictionaries and returns a `results` object containing
all logged time histories.

---
### Simulation Settings

In [ ]:
# ── Simulation Settings ───
# dt sets the master (inner loop) timestep.
# Middle and outer loops fire at 1/2 and 1/4 of this rate respectively.
# Reducing dt increases accuracy but slows down the simulation.

sim = {
    'dt'   : 0.005,              # master timestep [s] = inner loop rate (200 Hz)
    't_end': trajectory['t_end'] # end time [s]
}

---
## Section 5 — Example

The default parameters above represent a **lightweight indoor quadcopter** (1 kg, arm length
`l = 0.225 m` center-to-motor, giving a ~45 cm motor-to-motor diagonal in the plus configuration)
flying a moderately sized figure-8 at a gentle cruise speed.

Concretely:

- The trajectory spans **3 m × 2 m** in the horizontal plane (`A = 1.5 m`, `B = 1.0 m`).
- One full figure-8 cycle takes approximately **21 seconds** (`omega = 0.3 rad/s`, period ≈ 20.9 s).
- The vehicle completes roughly **two full cycles** in the 40-second simulation.
- Peak reference speed is **0.45 m/s** in X ($A \cdot \omega = 1.5 \times 0.3$) and
  **0.60 m/s** in Y ($2B\omega = 2 \times 1.0 \times 0.3$), both well within the near-linear regime.
- The 12-second soft-start ramp is included, so the first portion of the trajectory
  shows the vehicle accelerating from hover before locking onto the figure-8.

This is a well-conditioned baseline case: the vehicle is not pushed into large-angle flight,
making it a good starting point for verifying controller behavior before exploring
more aggressive trajectories.

Run the cell below to execute the simulation with these defaults.

In [ ]:
results = run_fig8_pid(vehicle, trajectory, controller, sim)
print('Simulation complete.')

---
## Section 6 — Results

### What to Expect

A well-tuned cascade PID on this trajectory should show:

- **Position time histories** — the actual X and Y signals closely follow the sinusoidal
  references after the soft-start transient (first ~12 s). Small amplitude and phase errors
  are normal; the vehicle cannot perfectly anticipate trajectory curvature.
- **Altitude** — Z should converge quickly and hold near `z_ref = 1.5 m` with very small
  deviation throughout, since the altitude loop is largely decoupled from horizontal motion.
- **Euler angles** — roll $\phi$ and pitch $\theta$ should remain small (within ±15° for
  this gentle trajectory), confirming the vehicle stays in the near-linear regime.
  Yaw $\psi$ should stay near zero since no yaw reference is commanded.
- **3D trajectory plot** — the steady-state path (after the soft-start) should trace
  a clean, repeating figure-8 with only slight rounding at the crossover and the tips.

### Plots

`plot_results(results)` generates three figures:

| Figure | Contents |
|--------|----------|
| **Position time history** | Actual vs reference for X, Y, Z over time |
| **Attitude time history** | Euler angles $\phi$, $\theta$, $\psi$ and body rates $p$, $q$, $r$ |
| **3D trajectory** | Actual path vs reference figure-8 (steady-state portion only) |

### Tracking Metrics

`compute_metrics(results)` reports RMSE values for X, Y, and Z tracking
over the steady-state window (after the soft-start).
Both absolute RMSE [m] and normalized RMSE (as a fraction of trajectory amplitude) are reported.

**Interpreting the metrics:**

- Normalized RMSE below **5 %** indicates excellent tracking for this trajectory type.
- Values between **5–15 %** suggest acceptable tracking with room for gain improvement.
- Values above **15 %** indicate the controller is struggling — check gain tuning
  starting from the inner loop, or reduce `omega` to slow the trajectory down.
- Z RMSE is typically an order of magnitude smaller than X/Y RMSE because altitude
  is decoupled from the lateral maneuvers.

### Does the Result Match Expected Behavior?

With the default parameters, the controller produces tracking errors consistent with
a well-tuned cascade PID on a gentle Lissajous trajectory:

- X and Y RMSE are in the range of **2–6 %** of their respective amplitudes.
- Altitude is maintained with RMSE well below 0.05 m.
- Roll and pitch angles remain within ±10°, confirming that the small-angle
  approximation used in the outer-loop decoupling is valid for this case.
- The 3D plot shows a clean, closed figure-8 in steady state.

These results confirm that the cascade PID with the given gains is correctly
regulating both position and attitude, and that the velocity feedforward is
effectively reducing the phase lag at the trajectory turns.

In [ ]:
# Time history plots — position, Euler angles, control inputs
# 3D trajectory plot (steady state)
plot_results(results)

In [ ]:
# RMSE metrics — absolute and normalized
metrics = compute_metrics(results)